In [ ]:
!pip install torch torchaudio --index-url https://download.pytorch.org/whl/cpu -q
!pip install transformers pdfplumber ipywidgets -q
!pip install --force-reinstall --no-deps "pillow==11.3.0" -q

import re
import io
import html

import torch
import torch.nn as nn

from transformers import AutoConfig, AutoModel, AutoTokenizer, PreTrainedModel

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

import pdfplumber


# Matches the detector's architecture: a DeBERTa backbone with mean pooling
# over the token embeddings, then a linear classifier head.
class AIDetectionModel(PreTrainedModel):
    config_class = AutoConfig
    _tied_weights_keys = []

    @property
    def all_tied_weights_keys(self):
        return {}

    def __init__(self, config):
        super().__init__(config)
        self.model = AutoModel.from_config(config)
        n = getattr(config, "detector_num_labels", 1)
        self.classifier = nn.Linear(config.hidden_size, n)

    def forward(self, input_ids, attention_mask=None, **kwargs):
        out = self.model(input_ids, attention_mask=attention_mask)
        h = out[0]
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (h * mask).sum(1) / mask.sum(1)
        return self.classifier(pooled)


repo = "wasitaigeneratedcom/ai-text-detector-small"

tokenizer = AutoTokenizer.from_pretrained(repo)
detector_model = AIDetectionModel.from_pretrained(repo).eval()

class_labels = ["human", "ai", "ai_edited", "humanized"]

# The model itself can handle up to 768 tokens per pass. For scoring the
# whole document we still chunk it, so mixed human/AI paragraphs don't get
# averaged into one blurry score. 300 tokens (roughly 210-220 words) sits
# between the 200 and 400 experiments, aiming for the sweet spot: enough
# context to stay reliable, small enough to still catch a switch between
# human and AI writing partway through a paragraph.
MODEL_MAX_TOKENS = 768
CHUNK_MAX_TOKENS = 300

# Most commercial detectors (Pangram, Originality.ai) refuse to score
# anything under ~50 words because short passages just don't carry enough
# signal. We set the bar a bit higher, at 100 words, since our chunking
# approach benefits from having a few chunks worth of text to average.
MIN_WORD_COUNT = 100


def get_scores(text):
    enc = tokenizer(text, truncation=True, max_length=768, return_tensors="pt")

    with torch.inference_mode():
        logits = detector_model(**enc)
        probs = torch.softmax(logits, dim=-1)[0]

    return {label: p.item() for label, p in zip(class_labels, probs)}


def normalize_text(text):
    return " ".join(text.split())


def word_count(text):
    return len(normalize_text(text).split())


def split_sentences(text):
    text = normalize_text(text)
    if not text:
        return []

    parts = re.split(r'(?<=[.!?])\s+(?=[A-Z0-9"\'])', text)
    return [p.strip() for p in parts if p.strip()]


def count_tokens(text):
    return len(tokenizer.encode(text, add_special_tokens=False))


def chunk_text_by_tokens(text, max_tokens=CHUNK_MAX_TOKENS):
    sentences = split_sentences(text)
    if not sentences:
        return []

    chunks = []
    current_sentences = []
    current_token_count = 0

    for sentence in sentences:
        sentence_token_count = count_tokens(sentence)

        if current_sentences and current_token_count + sentence_token_count > max_tokens:
            chunks.append((" ".join(current_sentences), current_token_count))
            current_sentences = [sentence]
            current_token_count = sentence_token_count
        else:
            current_sentences.append(sentence)
            current_token_count += sentence_token_count

    if current_sentences:
        chunks.append((" ".join(current_sentences), current_token_count))

    return chunks


def get_weighted_scores(text):
    chunks = chunk_text_by_tokens(text)
    if not chunks:
        return {label: 0.0 for label in class_labels}

    weighted_sums = {label: 0.0 for label in class_labels}
    total_weight = 0

    for chunk_text_piece, token_count in chunks:
        scores = get_scores(chunk_text_piece)
        weight = max(token_count, 1)
        total_weight += weight

        for label in class_labels:
            weighted_sums[label] += scores[label] * weight

    return {label: weighted_sums[label] / total_weight for label in class_labels}


# The highlighter uses the same 300-token window size as the document
# score (CHUNK_MAX_TOKENS), so the two numbers can't disagree about how
# much context counts as one "unit." Unlike the document score, the
# windows here slide one sentence at a time instead of sitting back to
# back, so a sentence is judged by every 300-token window it falls
# inside rather than just one fixed block. That's what keeps the
# highlighting smooth instead of chopping the text into blocky chunks.
HIGHLIGHT_COLORS = {
    "ai": "#ffb3b3",
    "ai_edited": "#ffe0b3",
    "humanized": "#e6ccff",
    "human": None,
}

# Stride of 1 checks every sentence as a window start, which is the most
# accurate but also the most model calls. Raising this to 2 or 3 skips
# some starting points to speed up very long documents, at a small cost
# in smoothness.
HIGHLIGHT_STRIDE = 1


def build_token_windows(sentences, target_tokens=CHUNK_MAX_TOKENS, stride=HIGHLIGHT_STRIDE):
    n = len(sentences)
    if n == 0:
        return []

    token_counts = [count_tokens(s) for s in sentences]
    windows = []

    for start in range(0, n, stride):
        total = 0
        end = start

        while end < n:
            next_total = total + token_counts[end]
            if end > start and next_total > target_tokens:
                break
            total = next_total
            end += 1

        windows.append((start, end))

    return windows


def analyze_spans(text):
    sentences = split_sentences(text)
    if not sentences:
        return []

    if len(sentences) == 1:
        scores = get_scores(sentences[0])
        top_label = max(scores, key=scores.get)
        return [{"text": sentences[0], "scores": scores, "label": top_label}]

    windows = build_token_windows(sentences)

    score_sums = [dict.fromkeys(class_labels, 0.0) for _ in sentences]
    window_counts = [0 for _ in sentences]

    for start, end in windows:
        chunk = " ".join(sentences[start:end])
        scores = get_scores(chunk)

        for i in range(start, end):
            for label in class_labels:
                score_sums[i][label] += scores[label]
            window_counts[i] += 1

    results = []
    for i, sentence in enumerate(sentences):
        count = max(window_counts[i], 1)
        avg_scores = {label: score_sums[i][label] / count for label in class_labels}
        top_label = max(avg_scores, key=avg_scores.get)
        results.append({"text": sentence, "scores": avg_scores, "label": top_label})

    return results


def create_highlighted_text(text):
    results = analyze_spans(text)
    if not results:
        return ""

    pieces = []
    for item in results:
        sentence = html.escape(item["text"])
        label = item["label"]
        bg = HIGHLIGHT_COLORS.get(label)

        if bg:
            score = item["scores"][label] * 100
            pieces.append(
                f'<span style="background:{bg}; border-radius:4px; padding:2px 3px;" '
                f'title="{label} score: {score:.1f}%">{sentence}</span>'
            )
        else:
            pieces.append(sentence)

    return " ".join(pieces)


text_box = widgets.Textarea(
    placeholder="Paste your text here...",
    layout=widgets.Layout(width="100%", height="150px")
)

pdf_upload = widgets.FileUpload(accept=".pdf", multiple=False, description="Upload PDF")
check_text_button = widgets.Button(description="Check Pasted Text", button_style="info")
check_pdf_button = widgets.Button(description="Check Uploaded PDF", button_style="warning")
output = widgets.Output()


def generate_report(text):
    text = normalize_text(text)

    if not text:
        print("No text found.")
        return

    wc = word_count(text)
    if wc < MIN_WORD_COUNT:
        display(HTML(
            f'<p style="font-family:Arial; color:#c0392b;">'
            f'This text is only {wc} words. Please enter at least '
            f'{MIN_WORD_COUNT} words so the model has enough context to '
            f'give a reliable result.</p>'
        ))
        return

    scores = get_weighted_scores(text)

    ai_score = round(
        min((scores["ai"] + scores["ai_edited"] + scores["humanized"]) * 100, 100),
        2
    )

    top_label = max(scores, key=scores.get)

    verdict_map = {
        "human": ("#27ae60", "Likely Human-Written"),
        "ai": ("#e74c3c", "Likely AI-Generated"),
        "ai_edited": ("#f39c12", "Human Text, AI-Polished"),
        "humanized": ("#8e44ad", "AI Text, Likely Paraphrased/Humanized"),
    }

    color, verdict = verdict_map[top_label]

    html_report = f"""
    <div style="border:2px solid {color}; border-radius:12px; padding:20px;
        font-family:Arial; max-width:700px; margin-bottom:20px;">
        <h2 style="color:{color}; margin:0 0 10px 0;">{verdict}</h2>
        <p style="font-size:16px; margin:0 0 10px 0;">
            AI involvement likelihood: <b>{ai_score}%</b>
        </p>
        <div style="background:#eee; border-radius:6px; height:18px; width:100%; margin-bottom:15px;">
            <div style="background:{color}; width:{ai_score}%; height:18px; border-radius:6px;"></div>
        </div>
        <div style="font-size:14px; line-height:1.6;">
            <b>Model scores</b><br>
            Human: {scores["human"] * 100:.1f}%<br>
            AI: {scores["ai"] * 100:.1f}%<br>
            AI edited: {scores["ai_edited"] * 100:.1f}%<br>
            Humanized: {scores["humanized"] * 100:.1f}%
        </div>
    </div>
    """

    display(HTML(html_report))

    display(HTML('<h3 style="font-family:Arial; margin-top:10px;">Potentially AI-like passages</h3>'))

    display(HTML(
        '<p style="font-family:Arial; font-size:13px; color:#666;">'
        f'Each sentence is scored by every {CHUNK_MAX_TOKENS}-token window it falls '
        'inside, then colored by whichever class scored highest on average. '
        '<span style="background:#ffb3b3; padding:1px 5px; border-radius:3px;">red</span> = AI-generated, '
        '<span style="background:#ffe0b3; padding:1px 5px; border-radius:3px;">orange</span> = AI-polished, '
        '<span style="background:#e6ccff; padding:1px 5px; border-radius:3px;">purple</span> = humanized/paraphrased. '
        'Unhighlighted text scored as human-written.'
        '</p>'
    ))

    placeholder = display(
        HTML('<p style="font-family:Arial; color:#888;">Please wait, scanning sentences...</p>'),
        display_id=True
    )

    highlighted = create_highlighted_text(text)

    placeholder.update(HTML(
        f'<div style="border:1px solid #ddd; border-radius:10px; padding:18px; '
        f'font-family:Georgia, serif; font-size:16px; line-height:1.7; '
        f'max-width:900px; background:#ffffff;">{highlighted}</div>'
    ))


def get_latest_pdf():
    value = pdf_upload.value
    if not value:
        return None

    if isinstance(value, dict):
        return list(value.values())[-1]

    return value[-1]


def on_check_text_clicked(b):
    with output:
        clear_output()

        if not text_box.value.strip():
            print("Please paste some text first.")
            return

        print("Analyzing... this can take a while for long text.")
        text = text_box.value

    with output:
        clear_output(wait=True)
        generate_report(text)


def on_check_pdf_clicked(b):
    with output:
        clear_output()

        uploaded_file = get_latest_pdf()
        if uploaded_file is None:
            print("Please upload a PDF first.")
            return

        print("Extracting text and analyzing... this can take a while for long text.")
        file_content = uploaded_file["content"]
        pdf_text = ""

        with pdfplumber.open(io.BytesIO(file_content)) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    pdf_text += page_text + "\n"

    with output:
        clear_output(wait=True)

        if not pdf_text.strip():
            print("Could not extract text from this PDF. Make sure it has selectable text, not scanned images.")
        else:
            generate_report(pdf_text)


check_text_button.on_click(on_check_text_clicked)
check_pdf_button.on_click(on_check_pdf_clicked)

display(widgets.HTML("<h3>Paste Text</h3>"))
display(text_box)
display(check_text_button)

display(widgets.HTML("<h3>Or Upload a PDF</h3>"))
display(pdf_upload)
display(check_pdf_button)

display(output)

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

HTML(value='<h3>Paste Text</h3>')

Textarea(value='', layout=Layout(height='150px', width='100%'), placeholder='Paste your text here...')

Button(button_style='info', description='Check Pasted Text', style=ButtonStyle())

HTML(value='<h3>Or Upload a PDF</h3>')

FileUpload(value={}, accept='.pdf', description='Upload PDF')

Button(button_style='warning', description='Check Uploaded PDF', style=ButtonStyle())

Output()